# Predicting Survival in the ICU
### Load required libraries

In [ ]:
library(readr); library(dplyr); library(ggplot2)
library(repr); library(rpart); library(rpart.plot)
library(caret)
library(rattle)
# install.packages("randomForest")   # run once if needed
library(randomForest)
options(repr.plot.width=9, repr.plot.height=6)

### Load data, flag dangerous BP

In [ ]:
ICU <- read.csv("/Users/emilyahmad/DataScience/OM065-Data-Science/Data/ICU.csv")

ICU$DangerBP <- 0
ICU$DangerBP[ICU$SysBP>140] <- 1

### Check survival rate and missingness

In [ ]:
# Overall survival rate (about 80%)
table(ICU$Survive)
mean(ICU$Survive)

# Check for missing values
summary(is.na(ICU) == TRUE)

### Stratified 80/20 train/test split

In [ ]:
# ONE split used for every model below.
# factor() makes caret sample within each class so the survival
# rate is preserved in both sets.
set.seed(111)
prop.index <- createDataPartition(factor(ICU$Survive), p = 0.8,
                                  list = FALSE, times = 1)
ICU.train <- ICU[prop.index, ]
ICU.test  <- ICU[-prop.index, ]

# Check the survival rate is (almost) the same in all three
mean(ICU$Survive)
table(ICU.train$Survive); mean(ICU.train$Survive)
table(ICU.test$Survive);  mean(ICU.test$Survive)

## Decision Tree
### Fit full unpruned tree

In [ ]:
tree <- rpart(Survive ~ Age + Sex + Infection + Pulse + DangerBP + Emergency,
              data = ICU.train, method = "class", cp = 0.00001)

### Define reusable tree plotting function

In [ ]:
# Reusable tidy plotting function
#  type = 4        -> split rule written on every branch (no overlapping labels)
#  extra = 104     -> class probabilities + % of patients in each node
#  fallen.leaves   -> all leaves on one row at the bottom (no staircase)
#  box.palette     -> red = died (0), green = survived (1)
plot_tree <- function(model, main = "") {
  rpart.plot(model,
             type = 4, extra = 104,
             fallen.leaves = TRUE,
             box.palette = c("lightcoral", "lightgreen"),
             shadow.col = "gray85",
             branch.lty = 1, branch.lwd = 1.5,
             under = FALSE, tweak = 1.1,
             main = main)
}

### Plot the full tree

In [ ]:
par(mfrow = c(1, 1))
plot_tree(tree, "Full tree (cp = 0.00001)")

### Full tree in-sample accuracy

In [ ]:
# In-sample accuracy
predicted <- predict(tree, ICU.train, type = "class")
table(predicted, ICU.train$Survive)
mean(predicted == ICU.train$Survive)

### Full tree out-of-sample accuracy

In [ ]:
# Out-of-sample accuracy
test.prediction <- predict(tree, ICU.test, type = "class")
table(test.prediction, ICU.test$Survive)
mean(test.prediction == ICU.test$Survive, na.rm=TRUE)

### Fit and prune tree

In [ ]:
tree_pruned <- rpart(Survive ~ Age + Sex + Infection + Pulse + DangerBP + Emergency,
                     data = ICU.train, method = "class", cp = 0.01)
printcp(tree_pruned)
best_cp <- tree_pruned$cptable[which.min(tree_pruned$cptable[, "xerror"]), "CP"]
tree_pruned <- prune(tree_pruned, cp = best_cp)

### Cross-validation error vs. tree size

In [ ]:
# Cross-validation plot: shows how error changes with tree size
plotcp(tree_pruned)

### Plot and save pruned tree

In [ ]:
# rpart.plot errors if pruning leaves only the root node, so check first
if (nrow(tree_pruned$frame) > 1) {
  plot_tree(tree_pruned, "Pruned tree")

  # Save a high-resolution PNG for reports / slides
  png("icu_pruned_tree.png", width = 2400, height = 1600, res = 250)
  plot_tree(tree_pruned, "Pruned tree")
  dev.off()
} else {
  message("Pruning left only the root node. Try a smaller cp (e.g. 0.005) in the pruned rpart() call.")
}

### Pruned tree in-sample accuracy

In [ ]:
predicted <- predict(tree_pruned, ICU.train, type = "class")
table(predicted, ICU.train$Survive)
mean(predicted == ICU.train$Survive)

### Pruned tree out-of-sample accuracy

In [ ]:
test.prediction <- predict(tree_pruned, ICU.test, type = "class")
table(test.prediction, ICU.test$Survive)
mean(test.prediction == ICU.test$Survive)

### Baseline: always predict survival

In [ ]:
# Baseline: always predict "survives"
mean(ICU.test$Survive == 1)

## Logistic Regression
### Fit logistic regression model

In [ ]:
logistic <- glm(Survive ~ Age + Sex + Emergency + Infection + Pulse + DangerBP,
                data = ICU.train, family = binomial)
summary(logistic)

ICU.train$Predicted <- fitted(logistic)

### Effect of a poor threshold

In [ ]:
# A deliberately poor threshold, to show why it matters
threshold <- 0.99
ICU.train$PredictSurvive <- ifelse(ICU.train$Predicted >= threshold, 1, 0)
summary(ICU.train$Predicted)
cor(ICU.train$Survive, ICU.train$Predicted)
addmargins(table(ICU.train$Survive))
mean(ICU.train$PredictSurvive == ICU.train$Survive)

### Optimize threshold on training set

In [ ]:
n <- 100
accuracy <- c()
thresholds <- c()
for (i in 1:n) {   # starts at 1: R indices start at 1, so i = 0 was being dropped
  threshold <- i/n
  PredictSurvive <- ifelse(ICU.train$Predicted >= threshold, 1, 0)
  accuracy[i] <- mean(PredictSurvive == ICU.train$Survive)
  thresholds[i] <- threshold
}
df <- na.omit(data.frame(thresholds, accuracy))
df <- df[df$accuracy < 1, ]
threshold.opt <- df$thresholds[which.max(df$accuracy)]
threshold.opt

### Apply threshold to test set

In [ ]:
ICU.test$Predicted <- predict(logistic, newdata = ICU.test, type = "response")
ICU.test$PredictSurvive <- ifelse(ICU.test$Predicted >= threshold.opt, 1, 0)
t.test <- table(ICU.test$Survive, ICU.test$PredictSurvive)
addmargins(t.test)
mean(ICU.test$PredictSurvive == ICU.test$Survive)

## Random Forest
### Fit forest, plot variable importance

In [ ]:
set.seed(111)
my_forest <- randomForest(factor(Survive) ~ Age + Sex + Infection + SysBP + Pulse + Emergency,
                          data = ICU.train, importance = TRUE, ntree = 1000)
varImpPlot(my_forest)

### Forest out-of-sample accuracy

In [ ]:
# Out-of-sample accuracy
my_prediction <- predict(my_forest, ICU.test, type = "class")
addmargins(table(my_prediction, ICU.test$Survive))
round(mean(my_prediction == ICU.test$Survive, na.rm=TRUE), 3)

### Forest in-sample accuracy

In [ ]:
# In-sample accuracy
my_prediction_train <- predict(my_forest, ICU.train, type = "class")
round(mean(my_prediction_train == ICU.train$Survive, na.rm=TRUE), 3)

## Summary: Train vs. Test Accuracy

In [ ]:
# ---- Train vs. test accuracy summary ----
acc <- function(pred, actual) round(mean(pred == actual, na.rm = TRUE), 3)

logit.train.pred <- ifelse(ICU.train$Predicted >= threshold.opt, 1, 0)

accuracy_summary <- data.frame(
  Model = c("Full decision tree (cp = 0.00001)",
            "Pruned decision tree",
            paste0("Logistic regression (threshold = ", threshold.opt, ")"),
            "Random forest (1000 trees)",
            "Baseline: always predict survive"),
  Train_Accuracy = c(acc(predict(tree, ICU.train, type = "class"), ICU.train$Survive),
                     acc(predict(tree_pruned, ICU.train, type = "class"), ICU.train$Survive),
                     acc(logit.train.pred, ICU.train$Survive),
                     acc(predict(my_forest, ICU.train, type = "class"), ICU.train$Survive),
                     acc(1, ICU.train$Survive)),
  Test_Accuracy  = c(acc(predict(tree, ICU.test, type = "class"), ICU.test$Survive),
                     acc(predict(tree_pruned, ICU.test, type = "class"), ICU.test$Survive),
                     acc(ICU.test$PredictSurvive, ICU.test$Survive),
                     acc(predict(my_forest, ICU.test, type = "class"), ICU.test$Survive),
                     acc(1, ICU.test$Survive))
)
accuracy_summary

### Which model performed best?

**Training set (in-sample)**
- **Best: random forest (0.994).** It fits the training data almost perfectly.
- Next: full decision tree (0.825), then logistic regression (0.806).
- Pruned tree (0.800): same as the baseline, because pruning left only the root node.

**Test set (out-of-sample)**
- **Tied for best: random forest and full decision tree (0.825).** They are the only models that beat the 0.800 baseline, and only by 2.5 points (1 extra correct patient out of 40).
- Logistic regression (0.800) and pruned tree (0.800): no better than always predicting "survives".

**Overfitting (train minus test)**
- **Random forest has the biggest gap (0.994 vs 0.825).** It memorizes the training data, and much of that does not carry over to new patients.
- Full tree (0.825 vs 0.825) and logistic regression (0.806 vs 0.800): little or no gap, so they generalize consistently.

**Overall**
- On new patients, the **full decision tree** matches the random forest's test accuracy without overfitting, so it is the best choice here.
- No model does much better than the baseline. With an 80% survival rate and only 40 test patients, these differences are small and could change with a different train/test split.